## Aufgabe 1

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
from pathlib import Path
import random

random.seed(20260217)

synset_words_file='/Users/luca/Projects/ms-data-science/deep-learning/hw5/imagenethelpercode/synset_words.txt'
imagenet_path = "/Users/luca/Projects/ms-data-science/deep-learning/hw5/imagenet-val"
imagenet_meta_path = "/Users/luca/Projects/ms-data-science/deep-learning/hw5/val"


In [ ]:
weights = ResNet50_Weights.DEFAULT
preprocess = weights.transforms()
model = resnet50(weights=weights)
model.eval();

In [ ]:
import xml.etree.ElementTree as ET

def parsesynsetwords(filen):

  synsetstoclassdescriptions={}
  indicestosynsets={}
  synsetstoindices={}
  ct=-1
  with open(filen) as f:
    for line in f:
      if (len(line)> 5):
        z=line.strip().split()
        descr=''
        for i in range(1,len(z)):
          descr=descr+' '+z[i]
        
        ct+=1
        indicestosynsets[ct]=z[0]
        synsetstoindices[z[0]]=ct
        synsetstoclassdescriptions[z[0]]=descr[1:]
  return indicestosynsets,synsetstoindices,synsetstoclassdescriptions

def parseclasslabel(nm,synsetstoindices):  
  tree = ET.parse(nm)
  root = tree.getroot()

  lbset=set()
  
  for obj in root.findall('object'):
     for name in obj.findall('name'):
       #print name.text
       ind=synsetstoindices[name.text]
       firstname=name.text
       lbset.add(ind)
       
  if len(lbset)!=1:
    print     ('ERR: len(lbset)!=1',  len(lbset))
    exit()
    
  for s in lbset:
    label=  s
  return label,firstname

In [ ]:
image1 = f"{imagenet_path}/n01601694/ILSVRC2012_val_00017028.JPEG"
metadata1 = f'{imagenet_meta_path}/ILSVRC2012_val_00017028.xml'

image2 = f"{imagenet_path}/n01534433/ILSVRC2012_val_00001915.JPEG"
metadata2 = f'{imagenet_meta_path}/ILSVRC2012_val_00001915.xml'

image3 = f"{imagenet_path}/n01843065/ILSVRC2012_val_00007284.JPEG"
metadata3 = f'{imagenet_meta_path}/ILSVRC2012_val_00007284.xml'

def predict_image(img_path, meta_path, print_log=False):
  with Image.open(img_path) as img:
    # Apply inference preprocessing transforms
    batch = preprocess(img).unsqueeze(0)
    
    # Use the model and print the predicted category
    prediction = model(batch).squeeze(0).softmax(0)
    class_id = prediction.argmax().item()
    score = prediction[class_id].item()
    category_name = weights.meta["categories"][class_id]

    # Print the actual class label
    indicestosynsets,synsetstoindices,synsetstoclassdescr=parsesynsetwords(synset_words_file)
    label,firstname=parseclasslabel(meta_path, synsetstoindices)
    if print_log:
      print("actual:", synsetstoclassdescr[indicestosynsets[label]], "\n")
      print(f"prediction: {category_name}: {100 * score:.1f}%")

    return category_name == synsetstoclassdescr[indicestosynsets[label]].split(", ")[0]

predict_image(image1, metadata1, print_log=True)
predict_image(image2, metadata2, print_log=True)
predict_image(image3, metadata3, print_log=True)

actual: water ouzel, dipper 

prediction: water ouzel: 31.0%
actual: junco, snowbird 

prediction: junco: 51.5%
actual: jacamar 

prediction: jacamar: 55.6%


True

In [ ]:
image_files = list(Path(imagenet_path).glob("**/*.JPEG"))
image_files = random.sample(image_files, 500)
meta_files = [f"{imagenet_meta_path}/{img.stem}.xml" for img in image_files]

In [ ]:
tp = 0.
n = 0.

errors = []

for img, meta in zip(image_files, meta_files):
    try:
        if predict_image(img, meta):
            tp+=1.
        n +=1.
    except:
        errors.append(img)

tp/n

0.8278688524590164

In [ ]:
print(f"Unable to process {len(errors)} images")

Unable to process 12 images


## Aufgabe 3

In [10]:

#https://github.com/pytorch/examples/blob/master/mnist/main.py

import argparse
import torch

import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms

import torch.utils

import numpy as np


torch.manual_seed(3)

def train_epoch(model,  trainloader,  criterion, device, optimizer ):

    model.train() # IMPORTANT!!!
 
    losses = []
    for batch_idx, data in enumerate(trainloader):

        inputs=data[0].to(device)
        labels=data[1].to(device)
      
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad() #reset accumulated gradients  
        loss.backward() #compute new gradients
        optimizer.step() # apply new gradients to change model parameters

    return losses


def evaluate(model, dataloader, criterion, device):

    model.eval() # IMPORTANT!!!


    with torch.no_grad(): # do not record computations for computing the gradient
    
      datasize = 0
      accuracy = 0
      avgloss = 0
      for ctr, data in enumerate(dataloader):

          #print ('epoch at',len(dataloader.dataset), ctr)
          
          inputs = data[0].to(device)        
          outputs = model(inputs)

          labels = data[1]

          # computing some loss
          cpuout= outputs.to('cpu')
          if criterion is not None:
            curloss = criterion(cpuout, labels)
            avgloss = ( avgloss*datasize + curloss ) / ( datasize + inputs.shape[0])

          # for computing the accuracy
          labels = labels.float()
          _, preds = torch.max(cpuout, 1) # get predicted class 
          accuracy =  (  accuracy*datasize + torch.sum(preds == labels) ) / ( datasize + inputs.shape[0])
            
          datasize += inputs.shape[0] #update datasize used in accuracy comp
    
    if criterion is None:   
      avgloss = None
          
    return accuracy, avgloss


def train_modelcv(dataloader_cvtrain, dataloader_cvtest ,  model ,  criterion, optimizer, scheduler, num_epochs, device):

  best_measure = 0
  best_epoch =-1

  for epoch in range(num_epochs):
    print('Epoch {}/{}'.format(epoch, num_epochs - 1))
    print('-' * 10)

    losses=train_epoch(model,  dataloader_cvtrain,  criterion,  device , optimizer )
    #scheduler.step()
    measure,_ = evaluate(model, dataloader_cvtest, criterion = None, device = device)
    
    print(' perfmeasure', measure.item() )

    # store current parameters because they are the best or not?
    if measure > best_measure: # > or < depends on higher is better or lower is better?
      bestweights= model.state_dict()
      best_measure = measure
      best_epoch = epoch
      print('current best', measure.item(), ' at epoch ', best_epoch)

  return best_epoch, best_measure, bestweights

def run(model):


  #parameters
  batchsize=32
  maxnumepochs=3 

  #device=torch.device("cuda:0")
  device=torch.device("cpu")

  datatransforms = transforms.Compose(
  [
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
  ])

  ds={
    'trainval': datasets.FashionMNIST('../hw2/data', train=True, download=True, transform=datatransforms), 
    'test': datasets.FashionMNIST('../hw2/data', train=False, download=True, transform=datatransforms)  
  }

  dataloaders={

  'train':  torch.utils.data.DataLoader(ds['trainval'], batch_size=batchsize, shuffle=False, sampler=  torch.utils.data.sampler.SubsetRandomSampler(np.arange(50000)) ), 

  'val': torch.utils.data.DataLoader(ds['trainval'], batch_size=batchsize, shuffle=False, sampler=  torch.utils.data.sampler.SubsetRandomSampler(np.arange(50000,60000)) ),

  'test':  torch.utils.data.DataLoader(ds['test'], batch_size=batchsize, shuffle=False)  
  }

  # model
  model.to(device)
  
  #loss 
  loss = torch.nn.CrossEntropyLoss(weight=None, size_average=None, ignore_index=-100, reduce=None, reduction='mean')

  lrates=[0.01, 0.001]

  best_hyperparameter= None
  weights_chosen = None
  bestmeasure = None

  for lr in lrates: # try a few learning rates

    print('\n\n\n###################NEW RUN##################')
    print('############################################')
    print('############################################')
    

    #optimizer here, because of lr, 
    # applies the computed gradients to change the trainable parameters of the model. 
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9) # which parameters to optimize during training?

    # train on train and eval on val data 
    best_epoch, best_perfmeasure, bestweights = train_modelcv(dataloader_cvtrain = dataloaders['train'], dataloader_cvtest = dataloaders['val'] ,  model = model ,  criterion = loss , optimizer = optimizer, scheduler = None, num_epochs = maxnumepochs , device = device)

    if best_hyperparameter is None:
      best_hyperparameter = lr
      weights_chosen = bestweights
      bestmeasure = best_perfmeasure
    elif best_perfmeasure > bestmeasure:
      best_hyperparameter = lr
      weights_chosen = bestweights
      bestmeasure = best_perfmeasure

  # end of for loop over hyperparameters here!
  model.load_state_dict(weights_chosen)

  accuracy,_ = evaluate(model = model , dataloader  = dataloaders['test'], criterion = None, device = device)

  print('accuracy val',bestmeasure.item() , 'accuracy test',accuracy.item()  )


In [19]:
import torch.nn as nn

class areallyoldschoolneuralnet(torch.nn.Module):
  def __init__(self,numcl):
    super().__init__()

    self.features = nn.Sequential(
      nn.Conv2d(1, 64, kernel_size=5, stride=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2),

      nn.Conv2d(64, 128, kernel_size=5, stride=2),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2),

      nn.Flatten()
    )

    self.classifier = nn.Linear(512, numcl)

  def forward(self, x):
    x = self.features(x)
    logits = self.classifier(x)
    return logits

numcl = 10
model = areallyoldschoolneuralnet(numcl)
run(model)




###################NEW RUN##################
############################################
############################################
Epoch 0/2
----------
 perfmeasure 0.8486002087593079
current best 0.8486002087593079  at epoch  0
Epoch 1/2
----------
 perfmeasure 0.8725000023841858
current best 0.8725000023841858  at epoch  1
Epoch 2/2
----------
 perfmeasure 0.8813999891281128
current best 0.8813999891281128  at epoch  2



###################NEW RUN##################
############################################
############################################
Epoch 0/2
----------
 perfmeasure 0.9017000198364258
current best 0.9017000198364258  at epoch  0
Epoch 1/2
----------
 perfmeasure 0.9049999713897705
current best 0.9049999713897705  at epoch  1
Epoch 2/2
----------
 perfmeasure 0.9075000286102295
current best 0.9075000286102295  at epoch  2
accuracy val 0.9075000286102295 accuracy test 0.9014000296592712
